# Data preparation for Llama
Stage 2 & 3

## A Overview

In [ ]:
from pathlib import Path
import pandas as pd

from datasets import Dataset
from transformers import BertForSequenceClassification, BertTokenizer

import configuration

from src import hf_utils, setup
from src.models import bert

In [ ]:
bert_model_path = Path("..") / "models" / "BERT"/ "w1_o0.99888" / "1.0"
tokens_path = Path("../tokens/BERT/llama_pre_bert_set")
datasets_path = Path("..") / "data"/"splitted"

device = setup.setup_device_with_seeds()

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(bert_model_path)
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_path)
bert_model.to(device)

In [ ]:
df_pre_bert = pd.read_csv(datasets_path / "llama_pre_bert_sets.csv")

# For testing purposes, we can sample a smaller subset of the data
df_pre_bert = df_pre_bert.sample(n=100, random_state=setup.RANDOM_SEED).reset_index(drop=True)

In [ ]:
if tokens_path.exists():
    print("Loading tokenized datasets from disk...")
    pre_bert_tokenized = Dataset.load_from_disk(tokens_path)
else:
    
    ds_pre_bert = Dataset.from_pandas(df_pre_bert)
    
    # Check max token length for the 'tweet_text' column
    hf_utils.max_length_dist(df_pre_bert, 'tweet_text', bert_tokenizer)
    
    pre_bert_tokenized = hf_utils.tokenize(ds_pre_bert, bert_tokenizer, tokens_path, bert.format_dataset)

In [ ]:
predictions, confidences = bert.predict(bert_model, pre_bert_tokenized, device, confidence_scores=True)
bert.report_metrics(pre_bert_tokenized, predictions)

In [ ]:
df_pre_bert["predicted"] = predictions
df_pre_bert["confidence"] = confidences
df_pre_bert.head()

In [ ]:
df_failed_predictions = df_pre_bert[df_pre_bert["informative"] != df_pre_bert["predicted"]]
df_failed_predictions.head()

In [ ]:
threshold = 0.2
df_low_confidence = df_pre_bert[df_pre_bert["confidence"] < threshold]
df_low_confidence.head()

In [ ]:
df = pd.concat([df_failed_predictions, df_low_confidence]).drop_duplicates().reset_index(drop=True)
df.to_csv(datasets_path / "llama_set.csv", index=False)